In [ ]:
# sc_flow/backends/torch/nn/_ae.py (or user's own file)
import torch.nn as nn


class AutoencoderModule(nn.Module):
    """Simple MLP autoencoder."""

    def __init__(self, input_dim: int, hidden_dims: list[int], latent_dim: int, *args, **kwargs):
        super().__init__()
        # Encoder
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            prev = h
        layers.append(nn.Linear(prev, latent_dim))
        self.encoder = nn.Sequential(*layers)

        # Decoder
        layers = []
        prev = latent_dim
        for h in reversed(hidden_dims):
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            prev = h
        layers.append(nn.Linear(prev, input_dim))
        self.decoder = nn.Sequential(*layers)

    def forward(self, x):
        """"""  # noqa
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon

In [ ]:
# user_contrib/autoencoder.py
import torch
import torch.nn as nn
from sc_flow.methods import register_method
from sc_flow.backends.torch.nn._modules import BaseModule
from sc_flow.data._composite import MatchedDistributions


class AEBaseModule(BaseModule):
    """"""  # noqa

    def __init__(self, dims_registry, *args, **kwargs):
        super().__init__()
        self.input_dim = len(dims_registry.feature_names)
        self._make_modules()

    def _make_modules(self):
        self.ae = AutoencoderModule(self.input_dim, hidden_dims=[128, 64], latent_dim=16)

    def forward(self, x):
        """"""  # noqa
        return self.ae(x)

    @classmethod
    def init_from_dims_registry(cls, dims_registry, *args, **kwargs):
        """"""  # noqa
        return cls(dims_registry, *args, **kwargs)


@register_method("ae-new", backend="torch", category="general")
class AutoencoderMethod:
    """"""  # noqa

    module_cls = AEBaseModule

    @staticmethod
    def train_step(flow, matched_distr: MatchedDistributions):
        """Required for 'general' category. Returns a metrics dict."""
        # 'flow' is the TorchBaseMethod instance (has .module, ._optimization_manager, etc.)
        # Extract target data (the input we want to reconstruct)
        target_state = flow._extract_state_data(matched_distr.target_distribution.state_data)
        if target_state is None:
            return {"loss": 0.0}  # Should not happen

        # Forward pass
        recon = flow.module(target_state)
        loss = nn.functional.mse_loss(recon, target_state)

        # Backward pass via the optimization manager
        flow._optimization_manager.backward_pass(loss)

        return {"loss": loss.item()}

    @staticmethod
    def predict(flow, matched_distr: MatchedDistributions):
        """Required. Returns reconstructed data."""
        target_state = flow._extract_state_data(matched_distr.target_distribution.state_data)
        flow.module.eval()
        with torch.no_grad():
            recon = flow.module(target_state)
        return recon

In [3]:
from sc_flow import SCFlow
import anndata
import numpy as np

# Create synthetic AnnData
n_cells = 1000
n_genes = 50
X = np.random.randn(n_cells, n_genes)
adata = anndata.AnnData(X)

# Register data
SCFlow.register_adata(adata)

# Instantiate the autoencoder model
model = SCFlow(method_id="ae-new", backend="torch")

# Train for a few epochs (each train_step is one batch update)
model.train(adata, n_train_steps=500, train_batch_size=128)

# Get reconstructions
reconstructed = model.predict(adata)
print(reconstructed.shape)  # (n_cells, n_genes)

/Users/lorenzo.consoli/micromamba/envs/sc-flow-tools/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 1/1 [00:00<00:00, 238.39it/s]
| loss:0.9156143665313721 | : 100%|██████████| 500/500 [00:00<00:00, 904.65it/s]
Predicting: 100%|██████████| 1/1 [00:00<00:00, 709.70it/s]


(1000, 50)
